In [ ]:
# Set some global parameters and paths
import os
import csv
import matplotlib.pyplot as plt
import numpy as np
from math import log

DEBUG = False
RELOAD = False
BASE_DIR = os.path.abspath("")
ROOT_DIR = os.path.abspath(os.path.join(BASE_DIR, '../../'))
EXPERIMENTS_DIR = os.path.join(BASE_DIR, 'experiments')
RESULT_DIR = os.path.join(ROOT_DIR, 'results/lumi_mixer/cases')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

if DEBUG:
    print("Base Directory:", BASE_DIR)
    print("Root Directory:", ROOT_DIR)
    print("Experiments Directory:", EXPERIMENTS_DIR)
    print("Data Directory:", RESULT_DIR)
    print("Output Directory:", OUTPUT_DIR)

# Open an experiment, read the information there

Each experiment is a CSV file with headers reflecting the parameters used to
create the case files.

In [ ]:
def read_experiment_file(experiment_name) -> list:
    """
    Read the experiment file and return a list of experiments with their
    parameters.

    Args:
        file_path (str): Path to the experiment CSV file.
    Returns:
        list: List of dictionaries, each representing an experiment with its
        parameters.
    """
    file_path = os.path.join(EXPERIMENTS_DIR, experiment_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Experiment file not found: {file_path}")

    # Read the CSV file and parse its contents
    with open(os.path.join(file_path), 'r') as file:
        reader = csv.DictReader(file, delimiter=',', skipinitialspace=True)
        tmp = [row for row in reader]

    # Extract experiments and convert types
    experiments = []
    for row in tmp:
        experiment = {}
        for key, value in row.items():
            try:
                experiment[key] = int(value)
            except ValueError:
                try:
                    experiment[key] = float(value)
                except ValueError:
                    if value.lower() in ['true', 'false']:
                        experiment[key] = value.lower() == 'true'
                    else:
                        experiment[key] = value

        # Check if the experiment has been completed and set the status
        log_file = f"{experiment['case_name']}.log"
        experiment['log_file'] = os.path.join(RESULT_DIR,
                                              experiment['case_name'],
                                              log_file)
        if os.path.exists(experiment['log_file']):
            experiment['status'] = 'done'
        else:
            experiment['status'] = 'pending'

        experiments.append(experiment)

    return experiments


In [ ]:
if DEBUG:
    experiment_file = os.path.join(EXPERIMENTS_DIR, 'single_node_capacity.csv')
    experiment = read_experiment_file(experiment_file)
    print(f"Loaded {len(experiment)} experiments.")
    for exp in experiment:
        print(exp)


# Parse the log files to extract the performance metrics

In [ ]:
def read_rt_stats_from_log(experiment: dict) -> list:
    """
    Read the runtime statistics from the log file.
    The runtime statistics will be stored in the experiment dictionary
    under the key 'runtimes'.

    Args:
        log_path (str): Path to the log file.
    """

    if not os.path.exists(experiment['log_file']):
        raise FileNotFoundError(
            f"Log file not found: {experiment['log_file']}")

    with open(experiment['log_file'], 'r') as file:
        lines = file.readlines()

    # Look for the pattern
    # `------Runtime statistics------` followed by an empty line
    # Then read the following table until an empty line is found

    start_index = None
    for i, line in enumerate(lines):
        if '------Runtime statistics------' in line:
            start_index = i + 2  # Skip the next empty line
            break
    if start_index is None:
        raise ValueError(
            f"Runtime statistics section not found in log file:\n\t{experiment['log_file']}"
        )

    end_index = start_index
    while end_index < len(lines) and lines[end_index].strip():
        end_index += 1

    # Extract the relevant lines
    stats_lines = lines[start_index:end_index]

    # Parse the statistics lines into a structured format
    header = ["Region name", "Total time", "Avg time", "Range +/-"]

    data = {}
    for line in stats_lines[2:]:
        parts = line.strip().split()
        if len(parts) >= 4:
            region_name = " ".join(parts[:-3])
            total_time = float(parts[-3])
            avg_time = float(parts[-2])
            range_plus_minus = float(parts[-1])

            data[region_name] = {
                header[1]: total_time,
                header[2]: avg_time,
                header[3]: range_plus_minus
            }
    experiment['runtimes'] = data


In [ ]:
# Parse the log files to extract the performance metrics
if DEBUG:
    read_rt_stats_from_log(experiment[1])
    print("Parsed Statistics Data:")
    for region, times in experiment[1]['runtimes'].items():
        print(f"Region: '{region}'")
        for key, value in times.items():
            print(f"   '{key}': {value}")


In [ ]:
def extract_times(experiment: list,
                  region: str,
                  measure: str = 'Avg time') -> np.ndarray:
    """
    Extract specific timing data from the experiment's runtimes.

    Args:
        experiment (list): The experiment dictionary containing runtimes.
        region (str): The region name to extract.
        measure (str): The measure to extract (default is 'Avg Time').
                       Possible values:  'Total time' 'Avg. time' 'Range +/-'

    Returns:
        np.ndarray: Array of extracted times.
    """

    result = np.array([])
    for exp in experiment:
        if exp['status'] != 'done':
            continue

        if 'runtimes' not in exp:
            read_rt_stats_from_log(exp)

        if 'runtimes' in exp and region in exp['runtimes']:
            result = np.append(result, exp['runtimes'][region][measure])
        else:
            result = np.append(result, 0.0)

    return result

# Single node capacity analysis

Here we have investigated the effect of adding more elements onto a single node
of LUMi. This will help us choose the right mesh size for a given problem.

In [ ]:
single_node_capacity = read_experiment_file('single_node_capacity.csv')

In [ ]:
# Plot the timestep and adjoint timestep as a function of the number of elements
num_elements = np.array([])
timestep = np.array([])
adjoint_timestep = np.array([])
checkpoint_save = np.array([])
checkpoint_restore = np.array([])
optimizer_time = np.array([])

num_elements = np.array([
    exp['Nx'] * exp['Ny'] * exp['Nz'] for exp in single_node_capacity
    if exp['status'] == 'done'
])
num_checkpoints = np.array([
    exp['N_memory'] for exp in single_node_capacity if exp['status'] == 'done'
])
mesh_labels = np.array([
    f"{exp['Nx']}x{exp['Ny']}x{exp['Nz']} {exp['N_memory']}"
    for exp in single_node_capacity
])

timestep = extract_times(single_node_capacity, 'Time-Step')
adjoint_timestep = extract_times(single_node_capacity, 'Time-Step Adjoint')
checkpoint_save = extract_times(single_node_capacity, 'Checkpoint save')
checkpoint_restore = extract_times(single_node_capacity, 'Checkpoint restore')
optimizer_time = extract_times(single_node_capacity, 'Optimizer iteration',
                               'Total time')

# Sort the data by the x_axis
# sorted_indices = np.argsort(timestep / num_elements, stable=True)
# mesh_labels = mesh_labels[sorted_indices]
# timestep = timestep[sorted_indices]
# adjoint_timestep = adjoint_timestep[sorted_indices]
# checkpoint_save = checkpoint_save[sorted_indices]
# checkpoint_restore = checkpoint_restore[sorted_indices]
# optimizer_time = optimizer_time[sorted_indices]

[fig_single, ax_single] = plt.subplots(1, 3, figsize=(18, 6))
ax_single[0].set_title('Timestep and Adjoint Timestep Per element')
ax_single[0].plot(timestep / num_elements, marker='o', label='Timestep')
ax_single[0].plot(adjoint_timestep / num_elements,
                  marker='s',
                  label='Adjoint Timestep')

ax_single[1].set_title('Checkpoint save and restore Per element')
ax_single[1].plot(checkpoint_save / num_elements,
                  marker='o',
                  label='Checkpoint Save')
ax_single[1].plot(checkpoint_restore / num_elements,
                  marker='s',
                  label='Checkpoint Restore')

ax_single[2].set_title('Optimizer Time Per element')
ax_single[2].plot(optimizer_time / num_elements,
                  marker='o',
                  label='Optimizer Time')

#  Set xscale to be log2 and yscale to be log10
for ax in ax_single:
    ax.set_yscale('log', base=10)
    ax.set_xlabel('Number of Elements')
    ax.set_ylabel('Time per element (s)')
    ax.set_xticks(range(len(mesh_labels)))
    ax.set_xticklabels(mesh_labels,
                       rotation=45,
                       ha='right',
                       rotation_mode='anchor')
    ax.legend()
    ax.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()

# Checkpoints

We know a limiting factor is the checkpoints. This section will investigate the
behaviour as we use more checkpoints for a couple of mesh sizes.

In [ ]:
checkpoint_count = read_experiment_file('memory_checkpoints.csv')

In [ ]:
# Plot the checkpoint save and restore times as a function of the number of
# checkpoints.

num_memory = np.array(
    [exp['N_memory'] for exp in checkpoint_count if exp['status'] == 'done'])
checkpoint_save = extract_times(checkpoint_count, 'Checkpoint save',
                                'Total time')
checkpoint_restore = extract_times(checkpoint_count, 'Checkpoint restore',
                                   'Total time')

# Get the mesh size for the labels and group by mesh size
mesh_size = np.array([
    f"{exp['Nx']}x{exp['Ny']}x{exp['Nz']}" for exp in checkpoint_count
    if exp['status'] == 'done'
])
mesh_labels = np.array(list(set(mesh_size)))

# Plot the checkpoint save and restore times in two subfigures grouped by
# mesh size
[fig_checkpoint, ax_checkpoint] = plt.subplots(1, 2, figsize=(15, 6))

for mesh in mesh_labels:
    mask = mesh_size == mesh
    ax_checkpoint[0].plot(num_memory[mask],
                          checkpoint_save[mask],
                          marker='o',
                          label=f'Checkpoint Save {mesh}')
    ax_checkpoint[1].plot(num_memory[mask],
                          checkpoint_restore[mask],
                          marker='s',
                          label=f'Checkpoint Restore {mesh}')

for i, ax in enumerate(ax_checkpoint):
    ax.set_xscale('log', base=2)
    ax.set_yscale('log', base=10)
    ax.set_xlabel('Number of Checkpoints')
    ax.set_ylabel('Time (s)')
    ax.set_xticks(num_memory)
    ax.set_xticklabels(num_memory)
    if i == 0:
        ax.set_title('Checkpoint save times')
    else:
        ax.set_title('Checkpoint restore times')
    ax.legend()
    ax.grid(True, which="both", ls="--")

plt.tight_layout()

In [ ]:
checkpoint_optimizer_time = extract_times(checkpoint_count,
                                          'Optimizer iteration', 'Total time')
checkpoint_num_elements = np.array([
    exp['Nx'] * exp['Ny'] * exp['Nz'] for exp in checkpoint_count
    if exp['status'] == 'done'
])

plt.figure(figsize=(10, 6))

for mesh in mesh_labels:
    mask = mesh_size == mesh
    plt.plot(num_memory[mask],
             checkpoint_optimizer_time[mask] / checkpoint_num_elements[mask],
             marker='o',
             label=f'Optimizer Time {mesh}')

plt.yscale('log', base=10)
plt.xlabel('Number of Checkpoints')
plt.ylabel('Optimizer Time (s)')
plt.title('Optimizer Time vs Number of Checkpoints for Each Mesh Size')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()

# Initial Weak scaling

From the above it seems the 128x16x16 with 400 checkpoints is a good candidate
for a weak scaling study. To narrow down the mesh size we can do a weak scaling
study with a few different aspect ratios and checkpoint amounts to see how they
behave as the network is stressed.


In [ ]:
weak_400 = read_experiment_file('weak_scaling_400.csv')
weak_200 = read_experiment_file('weak_scaling_200.csv')
weak_100 = read_experiment_file('weak_scaling_100.csv')

In [ ]:
weak_400_nodes = np.array(
    [exp['nodes'] for exp in weak_400 if exp['status'] == 'done'])
weak_200_nodes = np.array(
    [exp['nodes'] for exp in weak_200 if exp['status'] == 'done'])
weak_100_nodes = np.array(
    [exp['nodes'] for exp in weak_100 if exp['status'] == 'done'])

weak_400_times = extract_times(weak_400, 'Optimizer iteration', 'Total time')
weak_200_times = extract_times(weak_200, 'Optimizer iteration', 'Total time')
weak_100_times = extract_times(weak_100, 'Optimizer iteration', 'Total time')

# Merge the node lists to make sure you have all
x_axis = np.unique(
    np.concatenate((weak_400_nodes, weak_200_nodes, weak_100_nodes)))

# Sort the nodes and times by nodes
sorted_indices_400 = np.argsort(weak_400_nodes, stable=True)
weak_400_nodes = weak_400_nodes[sorted_indices_400]
weak_400_times = weak_400_times[sorted_indices_400]

sorted_indices_200 = np.argsort(weak_200_nodes, stable=True)
weak_200_nodes = weak_200_nodes[sorted_indices_200]
weak_200_times = weak_200_times[sorted_indices_200]

sorted_indices_100 = np.argsort(weak_100_nodes, stable=True)
weak_100_nodes = weak_100_nodes[sorted_indices_100]
weak_100_times = weak_100_times[sorted_indices_100]

plt.figure(figsize=(10, 6))
plt.plot(weak_400_nodes,
         weak_400_times[0] / weak_400_times,
         marker='o',
         label='Weak Scaling Efficiency 400')
plt.plot(weak_200_nodes,
         weak_200_times[0] / weak_200_times,
         marker='s',
         label='Weak Scaling Efficiency 200')
plt.plot(weak_100_nodes,
         weak_100_times[0] / weak_100_times,
         marker='^',
         label='Weak Scaling Efficiency 100')

plt.xscale('log', base=2)
plt.xlabel('Number of Nodes')
plt.ylabel('Efficiency')
plt.xticks(x_axis, x_axis)
plt.title('Weak Scaling Efficiency')
plt.ylim(0, 1.1)
plt.axhline(y=1.0, color='r', linestyle='--', label='Ideal Efficiency')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()

# The actual weak scaling study

We have determined so far that a mesh size of 64x16x16 with 800 checkpoints is
the best candidate for a weak scaling study. 
However, we have not enabled the parallel filesystem. For now we go with this
and tune the file system striping to this case.

- Stripe count: 1, stripe size: 1MB - falls bellow 60% at 8 nodes.
- 

In [ ]:
weak_scaling_experiments = read_experiment_file('weak_scaling.csv')

In [ ]:
weak_scaling_nodes = np.array([
    exp['nodes'] for exp in weak_scaling_experiments if exp['status'] == 'done'
])
weak_scaling_times = extract_times(weak_scaling_experiments,
                                   'Optimizer iteration', 'Total time')
weak_scaling_time_step = extract_times(weak_scaling_experiments, 'Time-Step')
weak_scaling_time_step_adj = extract_times(weak_scaling_experiments,
                                           'Time-Step Adjoint')
weak_scaling_checkpoint_save = extract_times(weak_scaling_experiments,
                                             'Checkpoint save')
weak_scaling_checkpoint_restore = extract_times(weak_scaling_experiments,
                                                'Checkpoint restore')
weak_scaling_mma_time = extract_times(weak_scaling_experiments,
                                      'MMA KKT computation', 'Total time')

# Sort the nodes and times by nodes
sorted_indices = np.argsort(weak_scaling_nodes)
weak_scaling_nodes = weak_scaling_nodes[sorted_indices]
weak_scaling_times = weak_scaling_times[sorted_indices]
weak_scaling_time_step = weak_scaling_time_step[sorted_indices]

plt.figure(figsize=(10, 6))

plt.plot(weak_scaling_nodes,
         weak_scaling_times[0] / weak_scaling_times,
         marker='o',
         markersize=8,
         linewidth=4,
         label='Weak Scaling Efficiency')
plt.plot(weak_scaling_nodes,
         weak_scaling_time_step[0] / weak_scaling_time_step,
         marker='s',
         linestyle=':',
         label='Time-Step Efficiency')
plt.plot(weak_scaling_nodes,
         weak_scaling_checkpoint_save[0] / weak_scaling_checkpoint_save,
         marker='^',
         linestyle=':',
         label='Checkpoint Save Efficiency')
plt.plot(weak_scaling_nodes,
         weak_scaling_checkpoint_restore[0] / weak_scaling_checkpoint_restore,
         marker='x',
         linestyle=':',
         label='Checkpoint Restore Efficiency')
plt.plot(weak_scaling_nodes,
         weak_scaling_time_step_adj[0] / weak_scaling_time_step_adj,
         marker='d',
         linestyle=':',
         label='Adjoint Time-Step Efficiency')
plt.plot(weak_scaling_nodes,
         weak_scaling_mma_time[0] / weak_scaling_mma_time,
         marker='v',
         linestyle=':',
         label='MMA Update Efficiency')

plt.xscale('log', base=2)
plt.xlabel('Number of Nodes')
plt.ylabel('Efficiency')
plt.xticks(weak_scaling_nodes, weak_scaling_nodes)
plt.title('Weak Scaling Efficiency')
plt.ylim(0, 1.1)
plt.axhline(y=1.0, color='r', linestyle='--', label='Ideal Efficiency')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()

# Storage Testing


In [ ]:
storage_experiment = read_experiment_file('storage_test.csv')

In [ ]:
# Look the the result folder and determine the filesize of the checkpoints
for exp in storage_experiment:
    if exp['status'] != 'done':
        continue

    result_folder = os.path.join(RESULT_DIR, exp['case_name'], 'checkpoints')
    if not os.path.exists(result_folder):
        exp['checkpoint_size'] = 0.0
        continue

    total_size = 0
    num_files = 0
    for file in os.listdir(result_folder):
        if file.endswith('.chkp') or file.endswith('.h5'):
            file_path = os.path.join(result_folder, file)
            total_size += os.path.getsize(file_path)
            num_files += 1
    average_size = total_size / num_files if num_files > 0 else 0
    exp['checkpoint_size'] = average_size

storage_mesh_labels = np.array([
    f"{exp['Nx']}x{exp['Ny']}x{exp['Nz']}" for exp in storage_experiment
    if exp['status'] == 'done'
])
# Plot a line graph of checkpoint size vs the experiment index
# First line should be for all experiments on a single node
# Second line should be showing the experiments as a function of number of nodes

storage_checkpoint_size = np.array([
    exp['checkpoint_size'] for exp in storage_experiment
    if exp['status'] == 'done'
])
storage_num_nodes = np.array(
    [exp['nodes'] for exp in storage_experiment if exp['status'] == 'done'])
storage_num_elements = np.array([
    exp['Nx'] * exp['Ny'] * exp['Nz'] for exp in storage_experiment
    if exp['status'] == 'done'
])

storage_checkpoint_size /= storage_num_elements

storage_line_1 = storage_checkpoint_size[storage_num_nodes == 1]
storage_line_1_elements = storage_num_elements[storage_num_nodes == 1]
storage_line_2 = storage_checkpoint_size[storage_num_nodes > 1]
storage_line_2_elements = storage_num_elements[storage_num_nodes > 1]

# Estimate A and B in the model size = A * N + B
# where N is the number of elements
# using linear regression on the single node experiments
if len(storage_line_1_elements) >= 2:
    a1, b1 = np.polyfit(storage_line_1_elements, storage_line_1, 1)
    print(f"Estimated a1: {a1}, b1: {b1}")
if len(storage_line_2_elements) >= 2:
    a2, b2 = np.polyfit(storage_line_2_elements, storage_line_2, 1)
    print(f"Estimated a2: {a2}, b2: {b2}")

plt.figure(figsize=(10, 6))
plt.plot(storage_line_1_elements,
         storage_line_1,
         marker='o',
         label='Single Node Experiments')
plt.plot(storage_line_2_elements,
         storage_line_2,
         marker='s',
         label='Multi Node Experiments')
plt.yscale('log', base=2)
plt.xlabel('Experiment Index / Number of Nodes')
plt.ylabel('Checkpoint Size')
plt.title('Checkpoint Size vs Experiment Index / Number of Nodes')
plt.xticks(np.unique(
    np.concatenate((storage_line_1_elements, storage_line_2_elements))),
           storage_mesh_labels,
           ha='right',
           rotation_mode='anchor',
           rotation=45)
y_axis = [
    2**i for i in range(int(log(min(storage_checkpoint_size), 2) - 1),
                        int(log(max(storage_checkpoint_size), 2)) + 1)
]
y_axis_label = []
for y in y_axis:
    if y < 1024:
        y_axis_label.append(f"{y:.0f}B")
    elif y < 1024 * 1024:
        y_axis_label.append(f"{y / 1024:.0f}KB")
    elif y < 1024 * 1024 * 1024:
        y_axis_label.append(f"{y / (1024*1024):.0f}MB")
    elif y < 1024 * 1024 * 1024 * 1024:
        y_axis_label.append(f"{y / (1024*1024*1024):.0f}GB")
    else:
        y_axis_label.append(f"{y / (1024*1024*1024*1024):.0f}TB")

plt.yticks(y_axis, y_axis_label)
plt.legend()
plt.grid(True, which="both", ls="--")
plt.tight_layout()
plt.show()